# Maia Corpus v9 — Acquisition Engine v0.4 (Clean)
**Google Colab · Google Drive persistente · Hugging Face isolado**

Este notebook **não altera as dependências do Python do Colab**. Usa um `venv` independente, executa um worker em subprocesso e grava os artefatos no Google Drive.

**Escopo:** piloto real de aquisição para matemática, código e literatura. O FineMath está excluído. Não modifica o corpus v8.

**Importante:** a seleção de fontes é configurável. A licença de cada registro deve ser verificada antes da incorporação ao treinamento. Registros sem licença aprovada ficam em quarentena.

## 1. Montar Google Drive
Execute em uma sessão Colab limpa. Não instale `datasets` ou `huggingface_hub` no Python principal.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_4_0/maia-corpus-v9')
assert Path('/content/drive/MyDrive').is_dir(), 'Google Drive não montado'
for folder in ['pilot/raw','pilot/accepted','pilot/quarantine','pilot/reports','pilot/logs']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
print('Destino persistente:', ROOT)

## 2. Configuração do piloto
Comece com limites pequenos. As fontes HF podem mudar de nome, esquema ou disponibilidade; o worker registra falhas por fonte sem interromper as demais.

- **Matemática:** OpenWebMath (amostra HF, sujeita à licença e acesso).
- **Código:** The Stack Smol (amostra HF, sujeita à licença e acesso).
- **Livros:** Gutenberg via URLs explícitas (não faz busca automática de obras).

**Nenhuma fonte é considerada licenciada automaticamente.**

In [ ]:
CONFIG = {
    'seed': 20261122,
    'max_documents_per_source': 100,
    'max_chars_per_document': 100_000,
    'sources': [
        {'name':'openwebmath', 'kind':'hf', 'domain':'math',
         'dataset':'open-web-math/open-web-math', 'split':'train', 'text_fields':['text','content'],
         'license_fields':['license','license_id','spdx_license'], 'enabled':True},
        {'name':'the_stack_smol', 'kind':'hf', 'domain':'code',
         'dataset':'bigcode/the-stack-smol', 'split':'train', 'text_fields':['content','text','code'],
         'license_fields':['license','license_id','spdx_license'], 'enabled':True},
        {'name':'gutenberg', 'kind':'gutenberg', 'domain':'books',
         'urls':[], 'enabled':False},
    ],
    'allowed_licenses': ['MIT','Apache-2.0','BSD-2-Clause','BSD-3-Clause','0BSD','Unlicense','CC0-1.0','CC-BY-4.0'],
}
import json
config_path = ROOT / 'pilot' / 'acquisition_config_v0_4.json'
config_path.write_text(json.dumps(CONFIG, ensure_ascii=False, indent=2), encoding='utf-8')
print(config_path)

## 3. Ambiente isolado
O ambiente é temporário no SSD do Colab, mas **todos os dados e relatórios ficam no Drive**. Não usamos `--system-site-packages` para impedir que pacotes do Colab contaminem o worker.

In [ ]:
import subprocess, sys
VENV = Path('/content/maia-v9-venv')
PY = VENV / 'bin/python'
if not PY.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(VENV)], check=True)
subprocess.run([str(PY), '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([str(PY), '-m', 'pip', 'install', '-q',
                'datasets>=3,<5', 'huggingface_hub>=0.30,<2', 'requests>=2.32,<3'], check=True)
subprocess.run([str(PY), '-c',
                'import datasets, huggingface_hub, requests; print("worker:",datasets.__version__,huggingface_hub.__version__,requests.__version__)'],
               check=True)
subprocess.run([sys.executable, '-c', 'import requests; print("Colab requests:",requests.__version__)'], check=True)

## 4. Worker de aquisição
O worker:
- transmite registros de HF via streaming (sem baixar o dataset inteiro);
- registra proveniência e SHA-256 do conteúdo;
- usa deduplicação exata dentro do piloto;
- separa **aceitos** e **quarentena** conforme licença por documento;
- limita documentos e caracteres;
- cria relatórios de sucesso/erro por fonte;
- grava arquivos no Drive, sem sobrescrever resultados anteriores (cada execução tem ID próprio).

**Atenção:** mesmo documentos marcados como aceitos exigem revisão de licenciamento, termos de uso e qualidade antes do treinamento.

In [ ]:
WORKER = '\nimport argparse, datetime, hashlib, json, re, traceback, urllib.request\nfrom pathlib import Path\nfrom datasets import load_dataset\n\ndef sha256(s):\n    return hashlib.sha256(s.encode("utf-8")).hexdigest()\n\ndef license_value(rec, fields):\n    for f in fields:\n        x=rec.get(f)\n        if isinstance(x,str) and x.strip(): return x.strip()\n    return None\n\ndef get_text(rec, fields):\n    for f in fields:\n        x=rec.get(f)\n        if isinstance(x,str) and x.strip(): return x\n    return None\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--root\',required=True)\n    ap.add_argument(\'--config\',required=True)\n    args=ap.parse_args()\n    root=Path(args.root)\n    cfg=json.loads(Path(args.config).read_text(encoding=\'utf-8\'))\n    run=datetime.datetime.now(datetime.timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\')\n    base=root/\'pilot\'\n    handles={}\n    def out(domain, status):\n        k=(domain,status)\n        if k not in handles:\n            target=base/status/f\'{domain}_{run}.jsonl\'\n            target.parent.mkdir(parents=True,exist_ok=True)\n            handles[k]=target.open(\'w\',encoding=\'utf-8\')\n        return handles[k]\n    seen=set()\n    summary=[]\n    maxdocs=int(cfg[\'max_documents_per_source\'])\n    maxchars=int(cfg[\'max_chars_per_document\'])\n    allowed=set(cfg[\'allowed_licenses\'])\n    for source in cfg[\'sources\']:\n        if not source.get(\'enabled\',False): continue\n        stats={\'source\':source[\'name\'],\'domain\':source[\'domain\'],\'examined\':0,\n               \'accepted\':0,\'quarantine\':0,\'duplicates\':0,\'error\':None}\n        try:\n            if source[\'kind\']==\'hf\':\n                ds=load_dataset(source[\'dataset\'], split=source.get(\'split\',\'train\'),\n                                streaming=True, trust_remote_code=False)\n                iterator=iter(ds)\n            elif source[\'kind\']==\'gutenberg\':\n                def gutenberg():\n                    for url in source.get(\'urls\',[]):\n                        if not url.startswith(\'https://www.gutenberg.org/\'):\n                            raise ValueError(\'Only gutenberg.org HTTPS URLs are allowed\')\n                        req=urllib.request.Request(url,headers={\'User-Agent\':\'MaiaCorpusV9Research/0.4\'})\n                        with urllib.request.urlopen(req,timeout=30) as response:\n                            data=response.read(2_000_001)\n                        if len(data)>2_000_000: raise ValueError(\'Gutenberg document exceeds 2MB\')\n                        yield {\'text\':data.decode(\'utf-8\',errors=\'replace\'),\'source_url\':url}\n                iterator=gutenberg()\n            else:\n                raise ValueError(\'Unknown kind\')\n            for rec in iterator:\n                if stats[\'examined\']>=maxdocs: break\n                stats[\'examined\']+=1\n                if not isinstance(rec,dict): continue\n                txt=get_text(rec,source.get(\'text_fields\',[\'text\']))\n                if not txt: continue\n                txt=txt[:maxchars]\n                digest=sha256(txt)\n                if digest in seen:\n                    stats[\'duplicates\']+=1\n                    continue\n                seen.add(digest)\n                lic=license_value(rec,source.get(\'license_fields\',[]))\n                status=\'accepted\' if lic in allowed else \'quarantine\'\n                row={\'domain\':source[\'domain\'],\'source\':source[\'name\'],\n                     \'dataset\':source.get(\'dataset\'),\'source_url\':rec.get(\'source_url\'),\n                     \'license\':lic,\'sha256\':digest,\'characters\':len(txt),\'text\':txt}\n                out(source[\'domain\'],status).write(json.dumps(row,ensure_ascii=False)+\'\\n\')\n                stats[status]+=1\n        except Exception as exc:\n            stats[\'error\']=f\'{type(exc).__name__}: {exc}\'\n            print(\'SOURCE ERROR\',source[\'name\'],stats[\'error\'],flush=True)\n        summary.append(stats)\n        print(json.dumps(stats,ensure_ascii=False),flush=True)\n    for h in handles.values(): h.close()\n    report={\'run_id\':run,\'created_utc\':datetime.datetime.now(datetime.timezone.utc).isoformat(),\n            \'configuration\':str(args.config),\'summary\':summary,\n            \'limitations\':[\'license metadata is not legal review\',\n                           \'exact dedup within run only\',\'no cross-v8 dedup\',\n                           \'not counted with MaiaTokenizer-65k\']}\n    dest=base/\'reports\'/f\'acquisition_report_{run}.json\'\n    dest.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    print(\'REPORT_PATH:\',dest,flush=True)\nif __name__==\'__main__\':\n    main()\n'
worker_path = ROOT / 'pilot' / 'maia_v9_worker_v0_4.py'
worker_path.write_text(WORKER, encoding='utf-8')
print('Worker:', worker_path)

## 5. Executar piloto
Se uma fonte não estiver acessível ou mudar de estrutura, o erro aparecerá no relatório. Não considere o piloto bem-sucedido apenas porque o processo retornou código zero.

In [ ]:
log_path = ROOT / 'pilot' / 'logs' / 'latest_worker.log'
with log_path.open('w', encoding='utf-8') as log:
    result = subprocess.run([str(PY), str(worker_path),
                             '--root', str(ROOT), '--config', str(config_path)],
                            stdout=log, stderr=subprocess.STDOUT, text=True)
print('Exit code:',result.returncode)
print(log_path.read_text(encoding='utf-8')[-12000:])
if result.returncode:
    raise RuntimeError('Worker failed. Inspect log.')

## 6. Inspecionar relatório e amostras
Um resultado com zero documentos pode significar fonte inacessível, exigência de autenticação, ausência de metadados de licença ou configuração incorreta. Confira os erros antes de escalar.

In [ ]:
reports = sorted((ROOT/'pilot'/'reports').glob('acquisition_report_*.json'))
assert reports, 'Nenhum relatório gerado'
report = json.loads(reports[-1].read_text(encoding='utf-8'))
for item in report['summary']:
    print(item)
print('\nRelatório:', reports[-1])
for domain in ('math','code','books'):
    for status in ('accepted','quarantine'):
        paths=sorted((ROOT/'pilot'/status).glob(f'{domain}_*.jsonl'))
        if paths:
            with paths[-1].open(encoding='utf-8') as f:
                line=f.readline()
            if line:
                item=json.loads(line)
                print(f'\n{domain}/{status}:', {'source':item['source'],'license':item['license'],
                                                  'characters':item['characters'],
                                                  'preview':item['text'][:250]})

## 7. Próximas decisões
1. Confirmar disponibilidade e termos de uso das fontes.
2. Revisar manualmente amostras e licenças, especialmente matemática e livros.
3. Definir fontes e cotas por idioma/domínio.
4. Construir amostra estratificada para o **MaiaTokenizer-65k**.
5. Escalar somente depois de validar qualidade, retomada, deduplicação e contagem real de tokens.

**Meta de planejamento:** 8B texto existente + 1,2B código + 600M matemática + 200M livros = ~10B tokens após retokenização.